# Configurable text / audio experiments: CV → submission → optional final model

1. Use **SHL Training (/venv/main)**. Edit the settings cell below; choose a unique experiment name whenever changing training/model/fold settings. The notebook runs one configuration across your chosen folds, not an automatic multi-model sweep.
2. Run the CV cells to obtain aligned OOF predictions, per-fold/pooled **RMSE**, a baseline, logs and the CV-ensemble submission CSV. Choose CrisperWhisper or original Whisper Large-v3 in TRANSCRIPT_DATASET. All original labels are retained, including the two empty Crisper transcripts.
3. The **last cell** lets you keep that CSV, choose the original fold checkpoint with the lowest validation RMSE, or train one fresh model on all labelled rows. Upload is disabled until you explicitly set `UPLOAD_ONE_MODEL=True`.
4. Download exactly submission.csv and run_metadata_<average_fold_rmse>.txt. The TXT is JSON containing full settings, best-epoch OOF rows, submission rows/CSV text, and encoded plots with their numerical data. Model files, tokenizer/config, predictions, metrics and logs stay on Vast after upload. No competition submission, Mac weight download or Vast shutdown happens automatically.


## 1. Edit all experiment hyperparameters here

In [ ]:
# ===== Experiment / backbone =====r
EXPERIMENT_NAME = 'firstaudrioronnlysimplemfflgp'  # NEW unique name when changing settings
TRANSCRIPT_DATASET = 'whisper'     # crisper | whisper (original Whisper Large-v3)
USE_EXISTING_RUN = False          # True loads that run's saved settings; ignores tweaks below

# ===== Modality selection =====
NUM_BACKBONES = 2                # 1 = text OR audio; 2 = text AND audio
SINGLE_BACKBONE_MODALITY = 'audio' # text | audio; ignored when NUM_BACKBONES = 2
# Missing pretrained text weights download automatically; cached weights are reused.
# Audio backbones below train on saved vectors; they need no pretrained download.

# ===== Backbone 1: text (ignored in audio-only mode) =====
# MODEL_NAME = 'microsoft/deberta-v3-large'
# MODEL_NAME = "microsoft/deberta-v3-base"
MODEL_NAME = "microsoft/deberta-v3-large"
# MODEL_NAME = "microsoft/deberta-v2-xlarge"
# MODEL_NAME = "FacebookAI/roberta-large"
# MODEL_NAME = "distilbert/distilbert-base-uncased"
# MODEL_NAME = "distilbert/distilbert-base-uncased"  ---- very bad gave cv of 0.4664
MODEL_REVISION = None             # Resolve and pin exact revision on first download; or specify SHA
# Encoder candidates from memory: microsoft/deberta-v3-base, microsoft/deberta-v3-large,
# microsoft/deberta-v2-xlarge, FacebookAI/roberta-large, distilbert/distilbert-base-uncased
PREPROCESSING = 'unchanged'       # Preserve grammar errors/disfluencies and all labels, including zeros

# ===== Audio inputs: actual fixed-channel names from the extraction schema =====
AUDIO_FEATURE_FILE = '/workspace/shl/audio_embeddings/shl_all_channels.h5'
# Change this path if your extracted HDF5 file is stored elsewhere on Vast.
# Add/remove entries below. Selected vectors concatenate; input width is inferred.
# To concatenate middle and final layers, select both named layer channels.
AUDIO_ENCODINGS = [
    'wavlm_layer8_mean',
    'wavlm_layer4_mean',
    'wavlm_layer4_std',
    'wavlm_layer8_std',
    'wavlm_layer12_mean',
    # 'wavlm_layer12_std',  # Also complete; enable if wanted

    'hubert_layer4_mean',
    'hubert_layer12_mean',
    'hubert_layer12_std',

    'wav2vec2_layer4_mean',
    'wav2vec2_layer12_mean',
    'wav2vec2_layer12_std',
]
# Sequence channels (word_wavlm/chunk_wavlm/phoneme_probabilities) are not fixed vectors.
AUDIO_NORMALIZATION = 'standard' # standard | none; imputation is training-fold-only

# ===== Backbone 2: extracted audio features =====
# AUDIO_BACKBONE = 'residual_mlp'
# AUDIO_BACKBONE = 'mlp'          # Feed-forward network
AUDIO_BACKBONE = 'gated_mlp'    # Learned gates blend transformed features
AUDIO_HIDDEN_DIM = 512
AUDIO_NUM_LAYERS = 2
AUDIO_OUTPUT_DIM = 256
AUDIO_DROPOUT = 0.2
AUDIO_LR = 1e-4

# ===== Text + audio fusion (used only with two backbones) =====
FUSION = 'concat'
FUSION_HIDDEN_DIM = 256
FUSION_DROPOUT = 0.2

# ===== Architecture =====
POOLING = 'mean'                  # mean | cls | attention | concat | weighted_layer | gem | lstm
CONCAT_LAST_N_LAYERS = 15         # Concatenate masked means of last N transformer blocks
WEIGHTED_LAYER_START = 0          # Zero-based first transformer layer; learned softmax weights
GEM_INITIAL_P = 3.0               # Signed masked power mean, p > 1
GEM_LEARNABLE_P = True
GEM_EPSILON = 1e-6
LSTM_HIDDEN_SIZE = 128            # One or more bidirectional token LSTMs, then masked mean
LSTM_NUM_LAYERS = 1
FREEZE_TOP_N_LAYERS = 0           # Final N output-side blocks; embeddings/pool/head remain trainable
RE_INIT_TOP_N_LAYERS = 0          # Final N blocks reset using backbone initialisation rules
# Nonzero freeze and reinitialisation overlap: use separate trials, never silently freeze reset layers.
BACKBONE_DROPOUT = 0.0
HEAD_DROPOUT = 0.0
OUTPUT_MODE = 'bounded'           # bounded = sigmoid × 5; linear trial must still pass CSV bounds
PREDICTION_MAX = 5.0
LOSS = 'mse'                     # Training objective; validation/selection ALWAYS uses RMSE

# ===== Grouped validation and seeds =====
N_FOLDS = 3
FOLD_SEED = 42
MODEL_SEED = 42
MODEL_SEED_STRIDE = 1009          # Fold model seed = MODEL_SEED + stride × zero-based fold index
VALIDATION_POLICY = 'grouped_multilabel_score_bins'  # zero, (0,2], (2,3], (3,4], (4,5]

# ===== Sequence length / measured batches =====
MAX_LENGTH = None                # None: tokenizer analysis selects supported >=95%-coverage candidate
# Explicit 512 reproduces the first run. Same length for training, validation and test.
TRAIN_MICROBATCH = None           # None probes 8,4,2,1; explicit size is also GPU-tested
EVAL_BATCH = None                 # None probes 16,8,4,2,1
GRADIENT_ACCUMULATION = None      # None computes accumulation for effective batch below
EFFECTIVE_BATCH_SIZE = 16
GRADIENT_CHECKPOINTING = True
PRECISION = 'bf16'               # Measured GPU pipeline currently supports BF16

# ===== Optimizer / schedule / stopping =====
BACKBONE_LR = 2e-5
HEAD_LR = 1e-4
LAYER_LR_DECAY = 1.0              # 1 disables layer-wise LR decay
WEIGHT_DECAY = 0.01              # Biases and one-dimensional parameters excluded
ADAM_BETAS = (0.9, 0.999)
ADAM_EPSILON = 1e-8
WARMUP_RATIO = 0.10              # Linear warmup then linear decay
MAX_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 3
MIN_RMSE_IMPROVEMENT = 0.0        # 0: strictly lowest validation RMSE selects best checkpoint
MAX_GRAD_NORM = 10.0
MAX_RUNTIME_HOURS = None         # None = unlimited CV time; optional cap pauses at saved epochs
CPU_THREADS = 4
HEARTBEAT_SECONDS = 15           # Independent process heartbeat; five-minute progress warning

# ===== Optional full-data refit, used only if chosen in LAST cell =====
FULL_DATA_EPOCHS = None           # None = ceil(median(best CV epochs)); optional integer override
FULL_DATA_RUN_NAME = EXPERIMENT_NAME + '_full_data'
# Refit is fresh pretrained initialisation, fixed epochs, no held-out validation score.

# ===== Optional private Kaggle publication =====
KAGGLE_USERNAME = 'prabalsinghx'
UPDATE_EXISTING_DATASET = False
DATASET_SLUG = 'shl-' + EXPERIMENT_NAME.replace('_','-') + '-model'
# Hidden API-key prompt appears only when upload is enabled in the LAST cell.


**Definitions and scope:** mean uses the last layer; CLS uses the first attended token; attention learns token weights. Concat, weighted-layer, signed GeM and bidirectional LSTM use the definitions/settings above. Layer pooling excludes the embedding output. All attended special tokens participate in masked pooling. Recurrent dropout is zero. “Top” layers are final blocks nearest the output; invalid counts are rejected after the actual architecture is loaded. Native context is verified, never extended merely by increasing a number. No long-document chunking is implemented.

Choose one text or audio backbone, or fuse one of each. Audio uses selected fixed vectors from the extracted HDF5 file; several encodings share one audio backbone. Missing values are imputed and scaling is fitted on each training fold only, with transforms stored in the checkpoint. Audio-only skips text model downloads and tokenization but retains the same labels and grouped folds from the input bundle. Full-data refit fits transforms on all labelled rows. Raw WAV training and sequence-channel training are not implemented. Changing the fold count or fold seed creates a different validation protocol; compare ensembles on common folds. Keep preprocessing unchanged until a separate trial is specified.

## 2. Resolve paths, freeze settings and show the complete configuration

In [ ]:
import json, sys, os, signal, subprocess
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display, HTML
import importlib.util
ROOT = Path('/workspace/shl')
BASE_CODE = ROOT / 'experiments/vast_text'
PYTHON = '/venv/main/bin/python'
spec = importlib.util.spec_from_file_location('shl_notebook_controls', BASE_CODE / 'notebook_controls.py')
controls = importlib.util.module_from_spec(spec)
spec.loader.exec_module(controls)
settings = dict(num_backbones=NUM_BACKBONES, single_backbone_modality=SINGLE_BACKBONE_MODALITY,
    audio_feature_file=AUDIO_FEATURE_FILE, audio_encodings=list(AUDIO_ENCODINGS),
    audio_backbone=AUDIO_BACKBONE, audio_normalization=AUDIO_NORMALIZATION,
    audio_hidden_dim=AUDIO_HIDDEN_DIM, audio_num_layers=AUDIO_NUM_LAYERS,
    audio_output_dim=AUDIO_OUTPUT_DIM, audio_dropout=AUDIO_DROPOUT, audio_lr=AUDIO_LR,
    fusion=FUSION, fusion_hidden_dim=FUSION_HIDDEN_DIM, fusion_dropout=FUSION_DROPOUT,
    transcript_dataset=TRANSCRIPT_DATASET, model_id=MODEL_NAME, revision=MODEL_REVISION, preprocessing=PREPROCESSING,
    pooling=POOLING, concat_last_n_layers=CONCAT_LAST_N_LAYERS, weighted_layer_start=WEIGHTED_LAYER_START,
    gem_initial_p=GEM_INITIAL_P, gem_learnable_p=GEM_LEARNABLE_P, gem_epsilon=GEM_EPSILON,
    lstm_hidden_size=LSTM_HIDDEN_SIZE, lstm_num_layers=LSTM_NUM_LAYERS,
    freeze_top_n_layers=FREEZE_TOP_N_LAYERS, re_init_top_n_layers=RE_INIT_TOP_N_LAYERS,
    backbone_dropout=BACKBONE_DROPOUT, head_dropout=HEAD_DROPOUT, output_mode=OUTPUT_MODE,
    prediction_max=PREDICTION_MAX, loss=LOSS, n_folds=N_FOLDS, fold_seed=FOLD_SEED,
    seed=MODEL_SEED, model_seed_stride=MODEL_SEED_STRIDE, validation_policy=VALIDATION_POLICY,
    max_length=MAX_LENGTH, batch_size=TRAIN_MICROBATCH, eval_batch_size=EVAL_BATCH,
    gradient_accumulation=GRADIENT_ACCUMULATION, effective_batch_size=EFFECTIVE_BATCH_SIZE,
    gradient_checkpointing=GRADIENT_CHECKPOINTING, precision=PRECISION, encoder_lr=BACKBONE_LR,
    head_lr=HEAD_LR, layer_lr_decay=LAYER_LR_DECAY, weight_decay=WEIGHT_DECAY,
    adam_betas=list(ADAM_BETAS), adam_epsilon=ADAM_EPSILON, warmup_ratio=WARMUP_RATIO,
    epochs=MAX_EPOCHS, patience=EARLY_STOPPING_PATIENCE, min_improvement=MIN_RMSE_IMPROVEMENT,
    max_grad_norm=MAX_GRAD_NORM, max_runtime_hours=MAX_RUNTIME_HOURS,
    cpu_threads=CPU_THREADS, heartbeat_seconds=HEARTBEAT_SECONDS)
CODE, RUN, CONFIG_PATH, requested = controls.setup(ROOT, EXPERIMENT_NAME, settings, USE_EXISTING_RUN)
FULL_RUN = ROOT / 'runs' / FULL_DATA_RUN_NAME
if USE_EXISTING_RUN:
    print('Loaded saved configuration; top-cell model/training tweaks are ignored for this existing run.')
print('Transcript dataset:', requested.get('transcript_dataset', 'whisper'))
print('Run:', RUN, '\nImplementation:', CODE, '\nConfig:', CONFIG_PATH)
display(pd.DataFrame([{'parameter': k, 'value': json.dumps(v)} for k,v in sorted(requested.items())]))

def live(command, auth=None):
    process = subprocess.Popen(command, stdin=subprocess.PIPE if auth is not None else None,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
    try:
        if auth is not None:
            process.stdin.write(auth + '\n'); process.stdin.close(); auth = None
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
        if code:
            raise RuntimeError(f'Command failed with exit code {code}; inspect its persistent logs.')
    except KeyboardInterrupt:
        os.killpg(process.pid, signal.SIGINT)
        try: process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGTERM); process.wait(timeout=30)
        raise

def download_link(path, label):
    path = Path(path)
    display(HTML(f'<a href="/files{path}" download>{label}</a>'))


## 3. Run or resume CV: automatic preparation, GPU checks and live logs

In [ ]:
# This cell downloads/prepares/trains the selected experiment. No publication occurs.
live([PYTHON, '-u', str(CODE / 'runner.py'), '--output', str(RUN), '--config', str(CONFIG_PATH)])
if not (RUN / 'metrics.json').exists():
    raise RuntimeError('CV is paused or incomplete. Resume this cell before reading final metrics.')
print('CV finished; submission ready:', RUN / 'submission.csv')


The runner verifies input checksums, pins the model revision, analyses the matching tokenizer, preserves verified source groups, measures two optimizer updates at worst-case length and runs the actual architecture/pooling GPU smoke test. Every run displays resolved batches, settings and trainable/frozen/reinitialised layer counts. Logs and independent 15-second heartbeats report stage/fold/epoch/batch, progress age, loss, LR, gradient norm, GPU utilisation/VRAM, disk and ETA. `train.log`, `events.jsonl`, `heartbeats.jsonl`, `latest_status.json` and `tensorboard/` persist in the run folder. A writer lock prevents concurrent launches. Exceptions remain visible. Interruptions resume from the last saved epoch; altered inputs/config/code/environment are rejected.

## 4. Resolved parameters, folds, token analysis and validation RMSE

In [ ]:
resolved = json.loads((RUN / 'resolved_config.json').read_text())
display(pd.DataFrame([{'parameter': k, 'value': json.dumps(v)} for k,v in sorted(resolved.items())]))
for name in ['length_report.json', 'fold_report.json', 'batch_measurements.json']:
    print('\n' + name); display(json.loads((RUN / name).read_text()))
records = []
for folder in sorted((RUN / 'models').glob('*/*/fold_*')):
    history = json.loads((folder / 'history.json').read_text())
    best = min(history, key=lambda r:r['validation_rmse'])
    fold = int(folder.name.split('_')[-1])
    records.append(dict(fold=fold+1, best_epoch=best['epoch'], epochs_completed=len(history),
                        validation_rmse=best['validation_rmse'], checkpoint=str(folder/'best.pt')))
fold_metrics = pd.DataFrame(records).sort_values('fold')
display(fold_metrics.drop(columns='checkpoint'))
print('Mean best-fold validation RMSE:', fold_metrics.validation_rmse.mean())
metrics = json.loads((RUN / 'metrics.json').read_text())
display({key:value for key,value in metrics.items() if key!='folds'})
print('Pooled OOF RMSE, zero/positive-label RMSE and same-fold training-mean baseline are separate from public leaderboard scores.')
print('Lowest-RMSE fold:', int(fold_metrics.loc[fold_metrics.validation_rmse.idxmin(),'fold']))


## 5. Review and download the original CV-ensemble submission

In [ ]:
submission = pd.read_csv(RUN / 'submission.csv')
expected = pd.read_csv(RUN / 'input/submission_template.csv')
assert list(submission.columns)==['filename','label'] and len(submission)==216 and submission.filename.is_unique
assert submission.filename.tolist()==expected.filename.tolist()
assert np.isfinite(submission.label).all() and submission.label.between(0,5).all()
display(submission.head(10))
print('The last cell exports the two download files for your chosen submission.')
print('Equally averages all', len(fold_metrics), 'fold predictions. No upload occurred.')


The official sample has 204 different IDs. This project exports the 216 actual test IDs in original order with `filename,label`, as established in the earlier submissions.

## 6. FINAL CHOICE: keep CSV, select lowest-RMSE fold, or refit on all data

1. `FINAL_ACTION='cv_submission'`: retain the original CV-ensemble CSV; no model training or upload.
2. `FINAL_ACTION='best_fold'`: use the original checkpoint with the lowest validation **RMSE**, create its own CSV, and optionally upload that one fold model. No refit.
3. `FINAL_ACTION='full_data'`: train/resume one fresh model on all 769 rows using the median best CV epoch count (or your top-cell override), create its final CSV, and optionally upload that one full-data model.

Set `UPLOAD_ONE_MODEL=True` only when you want publication. With `full_data`, keep it False initially to review the final CSV; rerunning after enabling upload verifies/reuses the finished refit instead of retraining. The full-data model has no independent validation RMSE; a CV-selected epoch count does not guarantee against overfitting. All local Vast artifacts remain saved after publication.

In [ ]:
# ===== Make your final choice here =====
FINAL_ACTION = 'cv_submission'    # cv_submission | best_fold | full_data
UPLOAD_ONE_MODEL = False          # Explicitly enable only after deciding to publish

if FINAL_ACTION not in {'cv_submission','best_fold','full_data'}:
    raise ValueError('Choose cv_submission, best_fold or full_data')
if FINAL_ACTION=='cv_submission' and UPLOAD_ONE_MODEL:
    raise ValueError('To upload one model, choose best_fold or full_data')

if FINAL_ACTION in {'cv_submission','best_fold'}:
    live([PYTHON, '-u', str(BASE_CODE/'notebook_controls.py'), '--code', str(CODE),
          '--run', str(RUN), '--choice', FINAL_ACTION])
    if FINAL_ACTION=='cv_submission':
        selected_submission = RUN/'submission.csv'
    else:
        choice = json.loads((RUN/'delivery/best_fold/selection.json').read_text())
        display(choice)
        selected_submission = Path(choice['submission'])
        print('Original lowest-RMSE fold model stays on Vast:', choice['local_model'])
else:
    command = [PYTHON, '-u', str(CODE/'full_data_train.py'), '--cv-run', str(RUN), '--output', str(FULL_RUN)]
    if FULL_DATA_EPOCHS is not None:
        command.extend(['--epochs', str(FULL_DATA_EPOCHS)])
    live(command)  # no publication in this command
    report = json.loads((FULL_RUN/'complete.json').read_text())
    plan = json.loads((FULL_RUN/'training_plan.json').read_text())
    display(plan)
    display(pd.DataFrame(report['history']))
    final_csv = pd.read_csv(FULL_RUN/'submission_full_data.csv')
    assert len(final_csv)==216 and final_csv.filename.tolist()==expected.filename.tolist()
    assert np.isfinite(final_csv.label).all() and final_csv.label.between(0,5).all()
    selected_submission = FULL_RUN/'submission_full_data.csv'
    print('Full-data model saved on Vast:', FULL_RUN/'final.pt')
    print('Full-data validation RMSE: unavailable; earlier CV evidence remains labelled separately.')

# Two downloadable outputs; OOF is always from each fold's selected best epoch.
sys.path.insert(0, str(BASE_CODE))
import importlib, run_metadata
importlib.reload(run_metadata)
submission_download, metadata_download = run_metadata.export(
    CODE, RUN, selected_submission, action=FINAL_ACTION,
    full_run=FULL_RUN if FINAL_ACTION=='full_data' else None)
download_link(submission_download, 'Download submission.csv')
download_link(metadata_download, 'Download ' + metadata_download.name)
print('Metadata contains both mean-fold and pooled OOF RMSE, full OOF/submission arrays, and encoded PNG plots.')
from IPython.display import Image
import base64
metadata = json.loads(metadata_download.read_text())
for name, plot in metadata['plots'].items():
    print(name)
    display(Image(data=base64.b64decode(plot['data'])))

if not UPLOAD_ONE_MODEL:
    print('Upload skipped. Your model, CSV and logs remain saved locally on Vast.')
else:
    import getpass
    from datetime import datetime, timezone
    dataset_id = KAGGLE_USERNAME + '/' + DATASET_SLUG + '-' + FINAL_ACTION.replace('_','-')
    destination = ROOT/'kaggle_publish'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
    if FINAL_ACTION=='best_fold':
        command = [PYTHON, '-u', str(BASE_CODE/'notebook_controls.py'), '--code', str(CODE), '--run', str(RUN),
                   '--choice', 'best_fold', '--publish', '--destination', str(destination), '--dataset-id', dataset_id]
    else:
        command = [PYTHON, '-u', str(CODE/'publish_full_model.py'), '--run', str(FULL_RUN),
                   '--destination', str(destination), '--dataset-id', dataset_id, '--auth-stdin']
    if UPDATE_EXISTING_DATASET: command.append('--update')
    print('Publishing ONE', FINAL_ACTION, 'model to PRIVATE Kaggle dataset:', dataset_id)
    key = getpass.getpass('Kaggle legacy API key (hidden; never saved): ')
    auth = json.dumps({'username':KAGGLE_USERNAME,'key':key}); del key
    try: live(command, auth=auth)
    finally: del auth
    print('Upload verified ready/private. Original model and all logs remain on Vast.')
    display(HTML(f'<a href="https://www.kaggle.com/datasets/{dataset_id}" target="_blank">Open private single-model dataset</a>'))
